In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time
import json
import re

In [2]:
options = Options()
# Tắt chế độ headless để bạn có thể tự tay giải Cloudflare nếu có yêu cầu
# options.add_argument("--headless=new") 
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

# Khởi tạo driver
driver = webdriver.Chrome(options=options)

# URL trang mục lục chính
main_url = "https://www.sanfoundry.com/1000-machine-learning-questions-answers/"
driver.get(main_url)

print("Chrome đã mở trang mục lục chính.")
print("Vui lòng giải Cloudflare thủ công trên trình duyệt (nếu có), sau đó tiếp tục chạy cell tiếp theo...")

Chrome đã mở trang mục lục chính.
Vui lòng giải Cloudflare thủ công trên trình duyệt (nếu có), sau đó tiếp tục chạy cell tiếp theo...


In [3]:
from urllib.parse import urljoin  # Thêm thư viện để chuẩn hóa link

# Lấy HTML của trang mục lục sau khi đã tải xong
html_main = driver.page_source
soup_main = BeautifulSoup(html_main, "lxml")

# Tìm khu vực chứa nội dung chính của bài viết
content_div = soup_main.find("div", class_="entry-content")

sub_urls = []
if content_div:
    # Tìm tất cả các thẻ <a> nằm trong các thẻ <li> (danh sách chủ đề)
    links = content_div.find_all("a")
    for link in links:
        href = link.get("href")
        
        # Nếu href tồn tại
        if href:
            # Tự động gộp đặt tên miền chính nếu href là link tương đối (ví dụ: /abc -> https://www.sanfoundry.com/abc)
            full_url = urljoin(main_url, href)
            
            # Lọc ra các link thuộc sanfoundry, tránh các link rác/quảng cáo và tránh trùng với trang chủ đề chính
            if "machine-learning" in full_url and full_url != main_url:
                sub_urls.append({
                    "title": link.get_text(strip=True),
                    "url": full_url
                })

print(f"Đã tìm thấy {len(sub_urls)} chủ đề con hợp lệ.")
# In thử 5 link đầu tiên để kiểm tra cấu trúc URL đã có đầy đủ "https://" chưa
for item in sub_urls[:5]:
    print(f"- {item['title']}: {item['url']}")

Đã tìm thấy 68 chủ đề con hợp lệ.
- Here are 1000 MCQs on Machine Learning (Chapterwise): https://www.sanfoundry.com/1000-machine-learning-questions-answers/#machine-learning-chapters
- Formal Learning Model: https://www.sanfoundry.com/1000-machine-learning-questions-answers/#formal-learning-model
- Version Spaces: https://www.sanfoundry.com/1000-machine-learning-questions-answers/#version-spaces
- VC-Dimension: https://www.sanfoundry.com/1000-machine-learning-questions-answers/#vc-dimension
- Linear Regression: https://www.sanfoundry.com/1000-machine-learning-questions-answers/#linear-regression


In [4]:
import re
import time
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By


def _classes_contain(tag, keyword):
    """Kiểm tra tag có class nào CHỨA keyword hay không (substring match).
    Lưu ý: tag.get("class") trả về 1 LIST các token class, ví dụ
    ["collapseomatic_content", "some-class"]. Dùng `keyword in list` sẽ
    kiểm tra bằng tuyệt đối (exact match) chứ KHÔNG phải substring -> đây
    chính là bug khiến bản cũ không bao giờ tìm thấy div đáp án."""
    classes = tag.get("class") or []
    return any(keyword in c for c in classes)


def crawl_questions_from_page(url, debug=False):
    driver.get(url)
    time.sleep(3)  # Chờ trang tải đầy đủ

    # --- BƯỚC 1: Tìm và click tất cả các nút "View Answer" ---
    # Ưu tiên các phần tử có class "collapseomatic" (nút bấm) nhưng KHÔNG
    # phải "collapseomatic_content" (chính là div đáp án ẩn) để tránh click
    # nhầm vào nội dung thay vì nút bấm.
    view_answer_buttons = driver.find_elements(
        By.XPATH,
        "//*[contains(@class, 'collapseomatic') and not(contains(@class, 'collapseomatic_content'))]"
    )
    # Fallback nếu class trên không khớp: tìm theo text "View Answer"
    if not view_answer_buttons:
        view_answer_buttons = driver.find_elements(
            By.XPATH, "//*[contains(text(), 'View Answer')]"
        )

    print(f"-> Đang mở {len(view_answer_buttons)} đáp án ẩn tìm thấy trên trang...")
    for btn in view_answer_buttons:
        try:
            driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", btn)
            time.sleep(0.05)
            driver.execute_script("arguments[0].click();", btn)
        except Exception:
            pass

    time.sleep(1.5)  # Chờ hiệu ứng hiển thị hoàn tất

    # --- BƯỚC 2: Bóc tách dữ liệu ---
    soup = BeautifulSoup(driver.page_source, "lxml")
    entry_content = soup.find("div", class_="entry-content")

    questions_data = []
    if not entry_content:
        return questions_data

    if debug:
        # In ra 1 đoạn HTML thô để kiểm tra cấu trúc thật (class name, cấp lồng...)
        print(entry_content.prettify()[:3000])

    elements = entry_content.find_all(["p", "div"], recursive=False)
    current_question = None

    for element in elements:
        text = element.get_text(" ", strip=True)
        if not text:
            continue

        # 1. Nhận biết câu hỏi mới (bắt đầu bằng số + dấu chấm, ví dụ "1. What is...")
        if element.name == "p" and re.match(r"^\d+\.", text):
            if current_question:
                questions_data.append(current_question)

            clean_text = text.replace("View Answer", "").strip()

            first_opt_match = re.search(r"\ba\)\s", clean_text)
            question_body = ""
            options_list = []

            if first_opt_match:
                split_idx = first_opt_match.start()
                question_body = clean_text[:split_idx].strip()
                options_part = clean_text[split_idx:].strip()

                opt_markers = list(re.finditer(r"\b([a-d])\)\s", options_part))
                for i in range(len(opt_markers)):
                    start = opt_markers[i].start()
                    end = opt_markers[i + 1].start() if i + 1 < len(opt_markers) else len(options_part)
                    options_list.append(options_part[start:end].strip())
            else:
                question_body = clean_text

            current_question = {
                "question": question_body,
                "options": options_list,
                "answer": "",
                "explanation": "",
            }

            # Fallback: nếu đáp án đúng được đánh dấu IN ĐẬM ngay trong câu hỏi
            # (thẻ <b>/<strong> bọc quanh phương án đúng) thay vì có dòng
            # "Answer: x" riêng, bắt lấy luôn ở đây.
            bold_tag = element.find(["b", "strong"])
            if bold_tag:
                btxt = bold_tag.get_text(" ", strip=True)
                m = re.match(r"^([a-dA-D])\)", btxt)
                if m:
                    current_question["answer"] = m.group(1).lower()

        # 2. Nếu các phương án lựa chọn nằm ở các thẻ <p> độc lập tiếp theo
        elif element.name == "p" and current_question and re.match(r"^[a-d]\)", text):
            clean_opt = text.replace("View Answer", "").strip()
            if clean_opt not in current_question["options"]:
                current_question["options"].append(clean_opt)

            bold_tag = element.find(["b", "strong"])
            if bold_tag and not current_question["answer"]:
                m = re.match(r"^([a-dA-D])\)", clean_opt)
                if m:
                    current_question["answer"] = m.group(1).lower()

        # 3. Lấy đáp án và giải thích từ khối đáp án ẩn (đã hiện ra sau khi click)
        #    FIX: dùng substring match qua _classes_contain() thay vì "in list"
        #    (bug cũ khiến điều kiện này không bao giờ đúng).
        elif element.name == "div" and current_question and (
            _classes_contain(element, "collapse") or _classes_contain(element, "sf-show")
        ):
            lines = [line.strip() for line in element.get_text("\n").split("\n") if line.strip()]
            for line in lines:
                low = line.lower()
                if low.startswith("answer:"):
                    current_question["answer"] = line.split(":", 1)[1].strip()
                elif low.startswith("explanation:"):
                    current_question["explanation"] = line.split(":", 1)[1].strip()

            # Fallback: nếu vẫn không có "Answer:" dạng text, tìm thẻ in đậm
            # bên trong khối đáp án (đáp án đúng thường được bôi đậm).
            if not current_question["answer"]:
                bold_tag = element.find(["b", "strong"])
                if bold_tag:
                    btxt = bold_tag.get_text(" ", strip=True)
                    m = re.match(r"^([a-dA-D])\)", btxt)
                    current_question["answer"] = m.group(1).lower() if m else btxt

    if current_question:
        questions_data.append(current_question)

    return questions_data


In [5]:
if sub_urls:
    test_subject = sub_urls[0]
    print(f"Đang cào thử nghiệm chủ đề: {test_subject['title']}")
    print(f"Đường link: {test_subject['url']}")
    
    test_results = crawl_questions_from_page(test_subject["url"])
    
    print(f"\nTìm thấy {len(test_results)} câu hỏi trong trang này:")
    # Định dạng in đẹp JSON bằng tiếng Việt không bị lỗi font (ensure_ascii=False)
    print(json.dumps(test_results[:2], indent=4, ensure_ascii=False))
else:
    print("Vui lòng chạy lại Cell 4 để lấy danh sách link trước.")

Đang cào thử nghiệm chủ đề: Here are 1000 MCQs on Machine Learning (Chapterwise)
Đường link: https://www.sanfoundry.com/1000-machine-learning-questions-answers/#machine-learning-chapters


-> Đang mở 48 đáp án ẩn tìm thấy trên trang...

Tìm thấy 15 câu hỏi trong trang này:
[
    {
        "question": "1. What is Machine learning ?",
        "options": [
            "a) The selective acquisition of knowledge through the use of computer programs",
            "b) The selective acquisition of knowledge through the use of manual programs",
            "c) The autonomous acquisition of knowledge through the use of computer programs",
            "d) The autonomous acquisition of knowledge through the use of manual programs"
        ],
        "answer": "c",
        "explanation": "Machine learning is the autonomous acquisition of knowledge through the use of computer programs."
    },
    {
        "question": "2. K-Nearest Neighbors (KNN) is classified as what type of machine learning algorithm?",
        "options": [
            "a) Instance-based learning",
            "b) Parametric learning",
            "c) Non-parametric learning",
            "d) Model-based learning"

In [7]:
import pprint as pp
pp.pprint(test_results)

[{'answer': 'c',
  'explanation': 'Machine learning is the autonomous acquisition of knowledge '
                 'through the use of computer programs.',
  'options': ['a) The selective acquisition of knowledge through the use of '
              'computer programs',
              'b) The selective acquisition of knowledge through the use of '
              'manual programs',
              'c) The autonomous acquisition of knowledge through the use of '
              'computer programs',
              'd) The autonomous acquisition of knowledge through the use of '
              'manual programs'],
  'question': '1. What is Machine learning ?'},
 {'answer': 'a',
  'explanation': 'KNN doesn’t build a parametric model of the data. Instead, '
                 'it directly classifies new data points based on the k '
                 'nearest points in the training data.',
  'options': ['a) Instance-based learning',
              'b) Parametric learning',
              'c) Non-parametric le